# 📊 Análisis de Fraude — Transferencias a Terceros
**Scotiabank Perú · Prevención de Fraude · COD_TRX=40 · TIPO_CUENTA=1010**

---
> **Indicadores:** `F`=Fraude · `G`=Buena · `P`=Pendiente · `D`=Descarte · `N`=Normal (no alertada)  
> **Importante:** Las métricas de Fraud Rate se calculan sobre clasificadas (F+G+D).  
> Las reglas se evalúan sobre **todo el volumen** incluyendo N.

## 0. Librerías y Configuración

In [ ]:
# ── Instalación silenciosa si falta algún paquete ─────────────────────────────
import subprocess, sys
def instalar(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['polars', 'altair', 'vl-convert-python', 'scikit-learn',
            'networkx', 'pyarrow', 'openpyxl', 'itables', 'great_tables']:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        instalar(pkg)

# ── Imports ───────────────────────────────────────────────────────────────────
import polars as pl
import polars.selectors as cs
import pandas as pd
import numpy as np
import altair as alt
import networkx as nx
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

# ── Altair: habilitar renderer para Jupyter ───────────────────────────────────
alt.data_transformers.disable_max_rows()

# ── Paleta de colores ─────────────────────────────────────────────────────────
C_FRAUD  = '#c0392b'
C_OK     = '#2980b9'
C_ACCENT = '#e67e22'
C_MUTED  = '#7f8c8d'
C_BG     = '#f8f9fa'

# ── Tema Altair global ────────────────────────────────────────────────────────
TEMA = {
    'config': {
        'background': C_BG,
        'view': {'stroke': 'transparent', 'fill': 'white'},
        'axis': {'labelColor': '#333', 'titleColor': '#333',
                 'gridColor': '#dee2e6', 'domainColor': '#adb5bd'},
        'title': {'color': '#212529', 'fontSize': 14, 'fontWeight': 'bold'},
        'legend': {'labelColor': '#333', 'titleColor': '#333'},
        'mark': {'tooltip': True},
    }
}
alt.themes.register('fraude', lambda: TEMA)
alt.themes.enable('fraude')

print(f'Polars  {pl.__version__}')
print(f'Altair  {alt.__version__}')
print('✅ Librerías listas')

---
# 1. Ingesta — Excel de Monitor

In [ ]:
BASE_DIR = Path(r'C:\Users\s4930359\Data_Herramientas\data\transferencias_terceros')

ARCHIVOS = {
    'Enero' : BASE_DIR / 'monitor_transferencias_enero.xlsx',
    'Marzo' : BASE_DIR / 'monitor_transferencias_marzo.xlsx',
    'Abril' : BASE_DIR / 'monitor_transferencias_abril.xlsx',
    # 'Mes4' : BASE_DIR / 'monitor_transferencias_mes4.xlsx',
}

def leer_monitor_excel(path: Path, mes: str) -> pl.DataFrame:
    """Lee export de Monitor: skip=4, todo str, limpia nombres."""
    df_pd = pd.read_excel(path, skiprows=4, dtype=str, header=0, engine='openpyxl')
    # Limpieza de nombres: espacios→_, guiones→_, mayúsculas, chars raros fuera
    df_pd.columns = (
        df_pd.columns
        .str.strip()
        .str.upper()
        .str.replace(r'\s+', '_', regex=True)
        .str.replace(r'-', '_', regex=False)
        .str.replace(r'[^A-Z0-9_]', '', regex=True)
    )
    return pl.from_pandas(df_pd).with_columns(pl.lit(mes).alias('MES_ORIGEN'))

frames = [leer_monitor_excel(path, mes) for mes, path in ARCHIVOS.items()]
raw    = pl.concat(frames, how='diagonal_relaxed')

print(f'Shape raw: {raw.shape[0]:,} filas × {raw.shape[1]} columnas')
raw.head(3)

## 1.1 Inspección de columnas

In [ ]:
print('── Columnas disponibles ──────────────────────────────────')
for i, col in enumerate(raw.columns, 1):
    print(f'{i:>3}. {col}')

print('\n── Nulos por columna ─────────────────────────────────────')
nulos = (
    raw.select([pl.col(c).is_null().sum().alias(c) for c in raw.columns])
       .unpivot(variable_name='columna', value_name='nulos')
       .with_columns((pl.col('nulos') / raw.height * 100).round(2).alias('pct_nulo'))
       .sort('pct_nulo', descending=True)
       .filter(pl.col('nulos') > 0)
)
print(nulos)

## 1.2 Mapeo de Columnas

> ⚠️ **Completa la clave izquierda** con el nombre exacto que apareció arriba.

In [ ]:
COL_MAP = {
    # Nombre exacto en Monitor              : Alias interno
    'NOMBRE_EXACTO_EN_MONITOR'              : 'FECHA_TRX',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'HORA_TRX',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'MONTO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'INDICADOR',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'ALERTA',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CUENTA',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'ID_CLIENTE',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'PRIMER_NOMBRE_CLIENTE',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'APELLIDO_CLIENTE',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'NOMBRE_BENEFICIARIO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CUENTA_DESTINO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'HASH_DISPOSITIVO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CANAL',
    'ACFTARJETA_RESGISTRO_750'              : 'TARJETA_REG',
    'ACFTARJETA_POS_76_DIGITOS'             : 'TARJETA_MID',
    'ACFSALDO_DISPONIBLE_EN_MONEDA_TRX'     : 'SALDO_DISPONIBLE_RAW',
    'CC_K05_COUNTTMP_TAMANO_COMERCIO'       : 'CNT_MES_PREVIO',
}

cols_ok  = {k: v for k, v in COL_MAP.items() if k in raw.columns}
cols_err = [k for k in COL_MAP if k not in raw.columns]

df = raw.rename(cols_ok)

if cols_err:
    print(f'⚠️  No encontradas (revisar nombre): {cols_err}')
else:
    print(f'✅ Todas las columnas mapeadas: {len(cols_ok)}')

## 1.3 Casteo de Tipos

In [ ]:
# ── 1. MONTO → Float (sin separadores) ───────────────────────────────────────
df = df.with_columns(
    pl.col('MONTO').cast(pl.Float64, strict=False)
)

# ── 2. SALDO DISPONIBLE → Float (punto como decimal) ─────────────────────────
df = (
    df.with_columns(
        pl.col('SALDO_DISPONIBLE_RAW')
          .cast(pl.Float64, strict=False)
          .alias('SALDO_DISPONIBLE')
    )
    .drop('SALDO_DISPONIBLE_RAW')
)

# ── 3. CNT_MES_PREVIO → Int ───────────────────────────────────────────────────
df = df.with_columns(
    pl.col('CNT_MES_PREVIO').cast(pl.Int32, strict=False)
)

# ── 4. FECHA_TRX → Date (AAAAMMDD pegado) ────────────────────────────────────
df = df.with_columns(
    pl.col('FECHA_TRX').str.strip_chars().str.to_date('%Y%m%d', strict=False)
)

# ── 5. HORA_TRX → Time (HH:MM:SS con dos puntos) ─────────────────────────────
df = df.with_columns(
    pl.col('HORA_TRX').str.strip_chars().str.to_time('%H:%M:%S', strict=False)
)

# ── 6. FECHA_HORA combinada → Datetime ───────────────────────────────────────
df = df.with_columns(
    (pl.col('FECHA_TRX').cast(pl.Utf8) + ' ' + pl.col('HORA_TRX').cast(pl.Utf8))
      .str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S', strict=False)
      .alias('FECHA_HORA')
)

# ── 7. TARJETA reconstruida ───────────────────────────────────────────────────
df = df.with_columns([
    (
        pl.col('TARJETA_REG').str.strip_chars().str.slice(0, 6) +
        pl.col('TARJETA_MID').str.strip_chars() +
        pl.col('TARJETA_REG').str.strip_chars().str.slice(-4)
    ).alias('TARJETA_FULL'),
    pl.col('TARJETA_REG').str.strip_chars().str.slice(0, 6).alias('BIN'),
    pl.col('TARJETA_REG').str.strip_chars().str.slice(-4).alias('TARJETA_LAST4'),
])

# ── 8. Limpiar strings clave ──────────────────────────────────────────────────
for col in ['INDICADOR', 'CANAL', 'NOMBRE_BENEFICIARIO', 'HASH_DISPOSITIVO']:
    if col in df.columns:
        df = df.with_columns(
            pl.col(col).str.strip_chars().str.to_uppercase().alias(col)
        )

print(f'✅ Casteo OK — {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.select(['FECHA_TRX','HORA_TRX','MONTO','SALDO_DISPONIBLE',
           'CNT_MES_PREVIO','TARJETA_FULL','BIN','CANAL']).head(5)

---
# 2. Ingeniería de Variables — Flags & Features

In [ ]:
# ── 2.1 Variable objetivo ─────────────────────────────────────────────────────
df = df.with_columns([
    (pl.col('INDICADOR') == 'F').cast(pl.Int8).alias('FLAG_FRAUDE'),
    pl.col('INDICADOR').is_in(['F','G','D']).cast(pl.Int8).alias('FLAG_CLASIFICADO'),
    (pl.col('INDICADOR') == 'P').cast(pl.Int8).alias('FLAG_PENDIENTE'),
])

# ── 2.2 Variables temporales ──────────────────────────────────────────────────
df = df.with_columns([
    pl.col('FECHA_TRX').dt.month().alias('MES'),
    pl.col('FECHA_TRX').dt.day().alias('DIA'),
    pl.col('FECHA_TRX').dt.weekday().alias('DIA_SEMANA'),
    pl.col('FECHA_HORA').dt.hour().alias('HORA'),
    pl.when(pl.col('FECHA_HORA').dt.hour().is_between(0,5)).then(pl.lit('Madrugada'))
      .when(pl.col('FECHA_HORA').dt.hour().is_between(6,11)).then(pl.lit('Mañana'))
      .when(pl.col('FECHA_HORA').dt.hour().is_between(12,17)).then(pl.lit('Tarde'))
      .otherwise(pl.lit('Noche')).alias('FRANJA_HORARIA'),
    (pl.col('FECHA_TRX').dt.weekday() >= 6).cast(pl.Int8).alias('FLAG_FIN_SEMANA'),
    (pl.col('FECHA_TRX').dt.day() >= 26).cast(pl.Int8).alias('FLAG_FIN_MES'),
])

# ── 2.3 Flags de monto ────────────────────────────────────────────────────────
df = df.with_columns([
    (pl.col('MONTO') >= 5000).cast(pl.Int8).alias('FLAG_MONTO_ALTO'),
    (pl.col('MONTO') >= 10000).cast(pl.Int8).alias('FLAG_MONTO_CRITICO'),
])

# ── 2.4 Saldo disponible ──────────────────────────────────────────────────────
df = (
    df.with_columns(
        (pl.col('MONTO') / (pl.col('SALDO_DISPONIBLE') + 1e-9))
          .round(4).alias('RATIO_MONTO_SALDO')
    )
    .with_columns([
        (pl.col('RATIO_MONTO_SALDO') >= 0.80).cast(pl.Int8).alias('FLAG_VACIADO_CUENTA'),
        (pl.col('SALDO_DISPONIBLE') < 200).cast(pl.Int8).alias('FLAG_SALDO_BAJO'),
    ])
)

# ── 2.5 Historia de cuenta (CC_K05) ──────────────────────────────────────────
df = df.with_columns([
    (pl.col('CNT_MES_PREVIO').is_null() | (pl.col('CNT_MES_PREVIO') == 0))
      .cast(pl.Int8).alias('FLAG_SIN_HISTORIA'),
    (pl.col('CNT_MES_PREVIO') <= 3)
      .cast(pl.Int8).alias('FLAG_CUENTA_NUEVA'),
])

# ── 2.6 Beneficiario nulo ─────────────────────────────────────────────────────
df = df.with_columns(
    (pl.col('NOMBRE_BENEFICIARIO').is_null() |
     (pl.col('NOMBRE_BENEFICIARIO').str.len_chars() == 0))
      .cast(pl.Int8).alias('FLAG_BENE_NULO')
)

# ── 2.7 Concentración de beneficiario ────────────────────────────────────────
conc_bene = (
    df.filter(pl.col('FLAG_BENE_NULO') == 0)
      .group_by('NOMBRE_BENEFICIARIO')
      .agg(pl.col('CUENTA').n_unique().alias('N_CUENTAS_ORIGEN_AL_BENE'))
)
df = df.join(conc_bene, on='NOMBRE_BENEFICIARIO', how='left')
df = df.with_columns(
    (pl.col('N_CUENTAS_ORIGEN_AL_BENE') >= 5).cast(pl.Int8).alias('FLAG_BENE_CONCENTRADO')
)

# ── 2.8 Hash dispositivo — ATO ────────────────────────────────────────────────
hash_multi = (
    df.filter(pl.col('HASH_DISPOSITIVO').is_not_null())
      .group_by('HASH_DISPOSITIVO')
      .agg(pl.col('CUENTA').n_unique().alias('N_CUENTAS_POR_DISPOSITIVO'))
)
df = df.join(hash_multi, on='HASH_DISPOSITIVO', how='left')
df = df.with_columns(
    (pl.col('N_CUENTAS_POR_DISPOSITIVO') >= 3).cast(pl.Int8).alias('FLAG_DISPOSITIVO_MULTIUSUARIO')
)

# ── 2.9 Velocidad por cuenta (diaria) ────────────────────────────────────────
vel_d = (
    df.group_by(['CUENTA','FECHA_TRX'])
      .agg([
          pl.len().alias('VEL_TRX_DIA_CUENTA'),
          pl.col('MONTO').sum().alias('VEL_MONTO_DIA_CUENTA'),
      ])
)
df = df.join(vel_d, on=['CUENTA','FECHA_TRX'], how='left')
df = df.with_columns(
    (pl.col('VEL_TRX_DIA_CUENTA') > 3).cast(pl.Int8).alias('FLAG_VEL_ALTA')
)

# ── 2.10 Z-score de monto por cuenta ─────────────────────────────────────────
stats_c = (
    df.group_by('CUENTA')
      .agg([pl.col('MONTO').mean().alias('_mu'), pl.col('MONTO').std().alias('_sd')])
)
df = (
    df.join(stats_c, on='CUENTA', how='left')
      .with_columns(
          ((pl.col('MONTO') - pl.col('_mu')) / (pl.col('_sd') + 1e-9))
            .round(3).alias('ZSCORE_MONTO_CUENTA')
      )
      .drop(['_mu','_sd'])
)
df = df.with_columns(
    (pl.col('ZSCORE_MONTO_CUENTA').abs() > 2.0).cast(pl.Int8).alias('FLAG_ANOMALIA_MONTO')
)

# ── Datasets derivados ────────────────────────────────────────────────────────
df_cal = df.filter(pl.col('FLAG_CLASIFICADO') == 1)   # F + G + D
df_f   = df.filter(pl.col('INDICADOR') == 'F')         # solo fraudes

print(f'df     : {df.shape[0]:>10,} filas  — todo el dataset')
print(f'df_cal : {df_cal.shape[0]:>10,} filas  — clasificadas (F+G+D)')
print(f'df_f   : {df_f.shape[0]:>10,} filas  — solo fraudes confirmados')

## 2.1 Resumen de Flags — ¿Cuáles discriminan mejor?

In [ ]:
FLAGS = [
    'FLAG_MONTO_ALTO','FLAG_MONTO_CRITICO','FLAG_VEL_ALTA',
    'FLAG_ANOMALIA_MONTO','FLAG_VACIADO_CUENTA','FLAG_SALDO_BAJO',
    'FLAG_CUENTA_NUEVA','FLAG_SIN_HISTORIA','FLAG_BENE_NULO',
    'FLAG_BENE_CONCENTRADO','FLAG_DISPOSITIVO_MULTIUSUARIO',
    'FLAG_FIN_SEMANA','FLAG_FIN_MES',
]
FLAGS = [f for f in FLAGS if f in df_cal.columns]

df_f_cal  = df_cal.filter(pl.col('FLAG_FRAUDE') == 1)
df_nf_cal = df_cal.filter(pl.col('FLAG_FRAUDE') == 0)

rows = []
for f in FLAGS:
    pf  = round(df_f_cal[f].mean() * 100, 2)
    pnf = round(df_nf_cal[f].mean() * 100, 2)
    rows.append({'Flag': f, 'Pct_Fraude': pf, 'Pct_NoFraude': pnf,
                 'Diferencia': round(pf - pnf, 2)})

resumen_flags = pl.DataFrame(rows).sort('Diferencia', descending=True)

chart_flags = (
    alt.Chart(resumen_flags.to_pandas())
    .transform_fold(['Pct_Fraude','Pct_NoFraude'], as_=['Grupo','Valor'])
    .mark_bar()
    .encode(
        y     = alt.Y('Flag:N', sort='-x', title=None),
        x     = alt.X('Valor:Q', title='% activación del flag'),
        color = alt.Color('Grupo:N',
                          scale=alt.Scale(domain=['Pct_Fraude','Pct_NoFraude'],
                                          range=[C_FRAUD, C_OK]),
                          legend=alt.Legend(title='Grupo')),
        column= alt.Column('Grupo:N', title=None),
        tooltip=['Flag:N','Grupo:N','Valor:Q']
    )
    .properties(width=300, height=280,
                title='% de activación por Flag — Fraude vs No Fraude')
)
chart_flags

---
# 3. KPIs Ejecutivos

In [ ]:
total_trx   = df.shape[0]
total_cal   = df_cal.shape[0]
total_fraud = df_f_cal.shape[0]
monto_fraud = df_f_cal['MONTO'].sum()
fraud_rate  = round(total_fraud / total_cal * 100, 4) if total_cal > 0 else 0
severidad   = round(monto_fraud / total_fraud, 2) if total_fraud > 0 else 0

print('━' * 65)
print(f'  Total transacciones (dataset)      : {total_trx:>15,}')
print(f'  Transacciones clasificadas (F+G+D) : {total_cal:>15,}')
print(f'  Fraudes confirmados (F)            : {total_fraud:>15,}')
print(f'  Monto total fraudulento (S/)       : {monto_fraud:>15,.2f}')
print(f'  Fraud Rate (sobre clasificadas)    : {fraud_rate:>14.4f}%')
print(f'  Severidad promedio (S/ por fraude) : {severidad:>15,.2f}')
print('━' * 65)

---
# 4. Análisis Univariado — Variables Continuas

## 4.1 MONTO — Distribución y densidad

In [ ]:
df_monto = (
    df_cal
    .filter(pl.col('MONTO').is_not_null())
    .with_columns(
        pl.when(pl.col('FLAG_FRAUDE')==1).then(pl.lit('Fraude'))
          .otherwise(pl.lit('No Fraude')).alias('GRUPO')
    )
    .select(['MONTO','GRUPO','INDICADOR'])
    .to_pandas()
)

# Densidad superpuesta Fraude vs No Fraude
density = (
    alt.Chart(df_monto)
    .transform_density('MONTO', as_=['MONTO','density'], groupby=['GRUPO'])
    .mark_area(opacity=0.45, line=True)
    .encode(
        x     = alt.X('MONTO:Q', title='Monto (S/)',
                       axis=alt.Axis(format=',.0f')),
        y     = alt.Y('density:Q', title='Densidad'),
        color = alt.Color('GRUPO:N',
                           scale=alt.Scale(domain=['Fraude','No Fraude'],
                                           range=[C_FRAUD, C_OK]),
                           legend=alt.Legend(title=None)),
        tooltip=['GRUPO:N', alt.Tooltip('MONTO:Q', format=',.0f')]
    )
    .properties(width=600, height=280,
                title='Densidad de Monto — Fraude vs No Fraude')
)

# Boxplot comparativo
boxplot = (
    alt.Chart(df_monto)
    .mark_boxplot(extent='min-max', size=40)
    .encode(
        x     = alt.X('GRUPO:N', title=None),
        y     = alt.Y('MONTO:Q', title='Monto (S/)',
                       axis=alt.Axis(format=',.0f')),
        color = alt.Color('GRUPO:N',
                           scale=alt.Scale(domain=['Fraude','No Fraude'],
                                           range=[C_FRAUD, C_OK]),
                           legend=None),
        tooltip=['GRUPO:N', alt.Tooltip('MONTO:Q', format=',.0f')]
    )
    .properties(width=250, height=280, title='Boxplot Monto')
)

(density | boxplot)

## 4.2 MONTO — Estadísticas descriptivas por indicador

In [ ]:
stats_monto = (
    df_cal.filter(pl.col('MONTO').is_not_null())
          .group_by('INDICADOR')
          .agg([
              pl.len().alias('N'),
              pl.col('MONTO').min().round(2).alias('Min'),
              pl.col('MONTO').quantile(0.25).round(2).alias('P25'),
              pl.col('MONTO').median().round(2).alias('Mediana'),
              pl.col('MONTO').mean().round(2).alias('Media'),
              pl.col('MONTO').quantile(0.75).round(2).alias('P75'),
              pl.col('MONTO').quantile(0.90).round(2).alias('P90'),
              pl.col('MONTO').max().round(2).alias('Max'),
              pl.col('MONTO').std().round(2).alias('SD'),
          ])
          .sort('INDICADOR')
)
print('Estadísticas de MONTO por Indicador:')
print(stats_monto)

## 4.3 MONTO — Discretización por 3 métodos

In [ ]:
from sklearn.tree import DecisionTreeClassifier

disc = (
    df_cal
    .filter(pl.col('MONTO').is_not_null() & pl.col('FLAG_FRAUDE').is_not_null())
    .select(['MONTO','FLAG_FRAUDE'])
    .to_pandas()
)

# ── MÉTODO 2: Árbol CART (recomendado) ───────────────────────────────────────
X = disc[['MONTO']].values
y = disc['FLAG_FRAUDE'].values

arbol = DecisionTreeClassifier(
    max_depth   = 3,
    min_samples_leaf = 10,
    class_weight = 'balanced'
)
arbol.fit(X, y)

# Extraer cortes del árbol
cortes_cart = sorted(set(
    arbol.tree_.threshold[arbol.tree_.threshold != -2]
))
print(f'📋 Cortes CART: {[round(c,2) for c in cortes_cart]}')

# Visualizar árbol
fig, ax = plt.subplots(figsize=(12, 4))
plot_tree(arbol, feature_names=['MONTO'], class_names=['No Fraude','Fraude'],
          filled=True, rounded=True, fontsize=9, ax=ax,
          impurity=False, proportion=True)
ax.set_title('Árbol CART — Discretización de MONTO', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

# Crear rangos
breaks = [-float('inf')] + cortes_cart + [float('inf')]
labels = [f'S/{breaks[i]:,.0f} - {breaks[i+1]:,.0f}' for i in range(len(breaks)-1)]
labels[-1] = f'S/{cortes_cart[-1]:,.0f}+'

disc['MONTO_CART'] = pd.cut(disc['MONTO'], bins=breaks, labels=labels, right=False)

# ── MÉTODO 1: Cuartiles ──────────────────────────────────────────────────────
q = disc['MONTO'].quantile([0.25,0.5,0.75]).tolist()
disc['MONTO_CUARTIL'] = pd.cut(
    disc['MONTO'],
    bins=[-float('inf')] + q + [float('inf')],
    labels=['Q1 Bajo','Q2 Med-Bajo','Q3 Med-Alto','Q4 Alto'],
    right=False
)

disc['GRUPO'] = disc['FLAG_FRAUDE'].map({0:'No Fraude', 1:'Fraude'})

# Gráficos comparativos
def grafico_prop(data, col_cat, titulo, orden=None):
    df_plot = (
        data.groupby([col_cat,'GRUPO'])
            .size().reset_index(name='N')
    )
    df_plot['TOT'] = df_plot.groupby(col_cat)['N'].transform('sum')
    df_plot['PCT'] = df_plot['N'] / df_plot['TOT']
    sort_order = orden or list(data[col_cat].cat.categories)
    return (
        alt.Chart(df_plot)
        .mark_bar()
        .encode(
            x     = alt.X(f'{col_cat}:N', sort=sort_order, title=None,
                           axis=alt.Axis(labelAngle=-30)),
            y     = alt.Y('PCT:Q', title='Proporción',
                           axis=alt.Axis(format='%')),
            color = alt.Color('GRUPO:N',
                               scale=alt.Scale(domain=['Fraude','No Fraude'],
                                               range=[C_FRAUD, C_OK]),
                               legend=alt.Legend(title=None)),
            tooltip=[f'{col_cat}:N','GRUPO:N',
                     alt.Tooltip('PCT:Q', format='.1%'),
                     alt.Tooltip('N:Q')]
        )
        .properties(width=260, height=240, title=titulo)
    )

g1 = grafico_prop(disc, 'MONTO_CUARTIL', 'Cuartiles')
g2 = grafico_prop(disc, 'MONTO_CART',    'Árbol CART ✅ Recomendado')
(g1 | g2)

## 4.4 Evaluación de la regla sobre TODO el volumen real (incluyendo N)

In [ ]:
# Aplicar cortes CART sobre df COMPLETO
df = df.with_columns(
    pl.col('MONTO').cut(
        breaks   = cortes_cart,
        labels   = labels,
        left_closed = True
    ).alias('MONTO_CAT_REGLA')
)

eval_real = (
    df.group_by('MONTO_CAT_REGLA')
      .agg([
          pl.len().alias('TOTAL'),
          (pl.col('INDICADOR')=='F').sum().alias('FRAUDES_F'),
          (pl.col('INDICADOR')=='G').sum().alias('BUENOS_G'),
          (pl.col('INDICADOR')=='N').sum().alias('NORMALES_N'),
          (pl.col('INDICADOR')=='P').sum().alias('PENDIENTES_P'),
          pl.col('MONTO').sum().round(2).alias('MONTO_TOTAL'),
          pl.col('MONTO').filter(pl.col('INDICADOR')=='F').sum().round(2).alias('MONTO_FRAUDE'),
      ])
      .with_columns([
          (pl.col('TOTAL') / df.shape[0] * 100).round(2).alias('PCT_VOLUMEN'),
          (pl.col('FRAUDES_F') / pl.col('TOTAL') * 100).round(4).alias('FRAUD_RATE_REAL'),
          (pl.col('FRAUDES_F') / total_fraud * 100).round(2).alias('PCT_FRAUDE_CAP'),
      ])
      .sort('MONTO_CAT_REGLA')
)

print('⚠️  Impacto REAL de la regla sobre el volumen completo (incluyendo N):')
print(eval_real)

# Gráfico de impacto real
eval_pd = eval_real.to_pandas()

bars_vol = (
    alt.Chart(eval_pd)
    .mark_bar(color=C_OK, opacity=0.6)
    .encode(
        x       = alt.X('MONTO_CAT_REGLA:N', title='Rango Monto',
                          axis=alt.Axis(labelAngle=-20)),
        y       = alt.Y('NORMALES_N:Q', title='N transacciones normales alertadas'),
        tooltip = ['MONTO_CAT_REGLA:N','TOTAL:Q','NORMALES_N:Q',
                   'FRAUDES_F:Q','PCT_VOLUMEN:Q']
    )
)
bars_fr = (
    alt.Chart(eval_pd)
    .mark_bar(color=C_FRAUD, opacity=0.9, size=20)
    .encode(
        x = alt.X('MONTO_CAT_REGLA:N'),
        y = alt.Y('FRAUDES_F:Q', title='Fraudes capturados')
    )
)
text_fr = (
    alt.Chart(eval_pd)
    .mark_text(color=C_FRAUD, dy=-8, fontWeight='bold')
    .encode(
        x    = alt.X('MONTO_CAT_REGLA:N'),
        y    = alt.Y('FRAUDES_F:Q'),
        text = alt.Text('MONTO_FRAUDE:Q', format=',.0f')
    )
)
(bars_vol + bars_fr + text_fr).properties(
    width=550, height=300,
    title='Impacto Real de Regla — Normales alertadas vs Fraudes capturados'
)

---
# 5. Análisis Univariado — Variables Categóricas

## 5.1 Indicador de Fraude

In [ ]:
tab_ind = (
    df.group_by('INDICADOR')
      .agg([
          pl.len().alias('N'),
          pl.col('MONTO').sum().round(2).alias('MONTO_TOTAL'),
          pl.col('MONTO').mean().round(2).alias('MONTO_PROM'),
      ])
      .with_columns((pl.col('N') / df.shape[0] * 100).round(2).alias('PCT'))
      .sort('N', descending=True)
)

chart_ind = (
    alt.Chart(tab_ind.to_pandas())
    .mark_bar()
    .encode(
        x     = alt.X('INDICADOR:N', sort='-y', title='Indicador'),
        y     = alt.Y('N:Q', title='N° Transacciones'),
        color = alt.Color('INDICADOR:N',
                           scale=alt.Scale(
                               domain=['F','G','N','P','D'],
                               range=[C_FRAUD, C_OK, C_MUTED, C_ACCENT, '#8e44ad']
                           ), legend=None),
        tooltip=['INDICADOR:N',
                 alt.Tooltip('N:Q', format=','),
                 alt.Tooltip('PCT:Q', format='.2f', title='%'),
                 alt.Tooltip('MONTO_PROM:Q', format=',.2f', title='Monto Prom S/')]
    )
    .properties(width=400, height=280,
                title='Distribución por Indicador de Fraude')
)
chart_ind

## 5.2 Canal ATM vs MOT — Heatmap + LIFT

In [ ]:
canal_franja = (
    df_cal
    .group_by(['CANAL','FRANJA_HORARIA'])
    .agg([
        pl.len().alias('TOTAL'),
        pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
    ])
    .with_columns([
        (pl.col('FRAUDES') / pl.col('TOTAL') * 100).round(3).alias('FRAUD_RATE'),
    ])
    .to_pandas()
)

heatmap = (
    alt.Chart(canal_franja)
    .mark_rect()
    .encode(
        x     = alt.X('FRANJA_HORARIA:N', title='Franja Horaria',
                       sort=['Madrugada','Mañana','Tarde','Noche']),
        y     = alt.Y('CANAL:N', title='Canal'),
        color = alt.Color('FRAUD_RATE:Q',
                           scale=alt.Scale(scheme='reds'),
                           title='Fraud Rate (%)'),
        tooltip=['CANAL:N','FRANJA_HORARIA:N',
                 alt.Tooltip('FRAUD_RATE:Q', format='.3f', title='Fraud Rate %'),
                 alt.Tooltip('FRAUDES:Q'), alt.Tooltip('TOTAL:Q')]
    )
    .properties(width=350, height=150,
                title='Fraud Rate: Canal × Franja Horaria')
)

texto_heat = (
    alt.Chart(canal_franja)
    .mark_text(fontWeight='bold')
    .encode(
        x    = alt.X('FRANJA_HORARIA:N', sort=['Madrugada','Mañana','Tarde','Noche']),
        y    = alt.Y('CANAL:N'),
        text = alt.Text('FRAUD_RATE:Q', format='.2f'),
        color= alt.condition(
            alt.datum.FRAUD_RATE > canal_franja['FRAUD_RATE'].median(),
            alt.value('white'), alt.value('#333')
        )
    )
)
(heatmap + texto_heat)

## 5.3 Franja Horaria con LIFT

In [ ]:
tasa_global = total_fraud / total_cal

tab_franja = (
    df_cal
    .group_by('FRANJA_HORARIA')
    .agg([
        pl.len().alias('TOTAL'),
        pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
        pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().round(2).alias('MONTO_FRAUDE'),
    ])
    .with_columns([
        (pl.col('FRAUDES') / pl.col('TOTAL') * 100).round(3).alias('FRAUD_RATE'),
        ((pl.col('FRAUDES') / pl.col('TOTAL')) / tasa_global).round(2).alias('LIFT'),
    ])
    .sort('FRAUD_RATE', descending=True)
)
print('LIFT > 1 = más fraude de lo esperado por azar')
print(tab_franja)

chart_franja = (
    alt.Chart(tab_franja.to_pandas())
    .mark_bar()
    .encode(
        y     = alt.Y('FRANJA_HORARIA:N', sort='-x', title=None),
        x     = alt.X('FRAUD_RATE:Q', title='Fraud Rate (%)'),
        color = alt.Color('LIFT:Q',
                           scale=alt.Scale(scheme='reds'),
                           title='LIFT'),
        tooltip=['FRANJA_HORARIA:N',
                 alt.Tooltip('FRAUD_RATE:Q', format='.3f'),
                 alt.Tooltip('LIFT:Q', format='.2f'),
                 alt.Tooltip('FRAUDES:Q'),
                 alt.Tooltip('MONTO_FRAUDE:Q', format=',.2f')]
    )
    .properties(width=450, height=200,
                title='Fraud Rate y LIFT por Franja Horaria')
)
chart_franja

---
# 6. Análisis Temporal — Evolución y Hora

In [ ]:
# Evolución mensual
evol = (
    df_cal
    .group_by('MES_ORIGEN')
    .agg([
        pl.len().alias('TOTAL_TRX'),
        pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
        pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().round(2).alias('MONTO_FRAUDE'),
    ])
    .with_columns([
        (pl.col('FRAUDES') / pl.col('TOTAL_TRX') * 100).round(4).alias('FRAUD_RATE'),
        (pl.col('MONTO_FRAUDE') / pl.col('FRAUDES')).round(2).alias('SEVERIDAD'),
    ])
    .sort('MES_ORIGEN')
    .to_pandas()
)

barras = (
    alt.Chart(evol)
    .mark_bar(color=C_FRAUD, opacity=0.8)
    .encode(
        x       = alt.X('MES_ORIGEN:N', title='Mes'),
        y       = alt.Y('FRAUDES:Q', title='N° Fraudes'),
        tooltip = ['MES_ORIGEN:N','FRAUDES:Q',
                   alt.Tooltip('FRAUD_RATE:Q', format='.4f', title='Fraud Rate %'),
                   alt.Tooltip('MONTO_FRAUDE:Q', format=',.2f', title='Monto S/')]
    )
)
linea_fr = (
    alt.Chart(evol)
    .mark_line(color=C_ACCENT, strokeWidth=2.5, point=True)
    .encode(
        x = alt.X('MES_ORIGEN:N'),
        y = alt.Y('FRAUD_RATE:Q', title='Fraud Rate (%)',
                   axis=alt.Axis(orient='right'))
    )
)

evol_chart = (
    alt.layer(barras, linea_fr)
    .resolve_scale(y='independent')
    .properties(width=500, height=280, title='Evolución Mensual — Fraudes y Fraud Rate')
)
evol_chart

In [ ]:
# Hora del día — volumen vs fraud rate
por_hora = (
    df_cal
    .group_by('HORA')
    .agg([pl.len().alias('TOTAL'), pl.col('FLAG_FRAUDE').sum().alias('FRAUDES')])
    .with_columns((pl.col('FRAUDES')/pl.col('TOTAL')*100).round(3).alias('FRAUD_RATE'))
    .sort('HORA')
    .to_pandas()
)

base_hora = alt.Chart(por_hora).encode(x=alt.X('HORA:O', title='Hora del día'))

vol_bar = base_hora.mark_bar(color=C_OK, opacity=0.5).encode(
    y=alt.Y('TOTAL:Q', title='Volumen transacciones'),
    tooltip=['HORA:O','TOTAL:Q','FRAUDES:Q',
             alt.Tooltip('FRAUD_RATE:Q', format='.3f', title='Fraud Rate %')]
)
fr_line = base_hora.mark_line(color=C_FRAUD, strokeWidth=2.5, point=True).encode(
    y=alt.Y('FRAUD_RATE:Q', title='Fraud Rate (%)', axis=alt.Axis(orient='right'))
)

(
    alt.layer(vol_bar, fr_line)
    .resolve_scale(y='independent')
    .properties(width=650, height=280,
                title='Volumen vs Fraud Rate por Hora del Día')
)

---
# 7. Ingeniería de Variables — Velocidad

In [ ]:
# Tabla resumen de velocidad por cuenta
vel_cuenta = (
    df
    .group_by('CUENTA')
    .agg([
        pl.len().alias('TOTAL_TRX'),
        pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
        (pl.col('INDICADOR')=='G').sum().alias('BUENOS_G'),
        (pl.col('INDICADOR')=='N').sum().alias('NORMALES_N'),
        (pl.col('INDICADOR')=='P').sum().alias('PENDIENTES_P'),
        (pl.col('INDICADOR')=='D').sum().alias('DESCARTE_D'),
        pl.col('MONTO').sum().round(2).alias('MONTO_TOTAL'),
        pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().round(2).alias('MONTO_FRAUDE'),
        pl.col('FECHA_TRX').n_unique().alias('DIAS_ACTIVO'),
        pl.col('CUENTA_DESTINO').n_unique().alias('N_DESTINOS_DISTINTOS'),
    ])
    .with_columns([
        (pl.col('FRAUDES') / pl.col('TOTAL_TRX') * 100).round(3).alias('FRAUD_RATE'),
        (pl.col('TOTAL_TRX') / (pl.col('DIAS_ACTIVO') + 1)).round(2).alias('TRX_POR_DIA'),
    ])
)

print(f'Cuentas distintas         : {vel_cuenta.shape[0]:,}')
print(f'Cuentas con al menos 1 F  : {vel_cuenta.filter(pl.col("FRAUDES")>0).shape[0]:,}')
print('\nTop 20 cuentas con mayor monto fraudulento:')
print(
    vel_cuenta
    .filter(pl.col('FRAUDES') > 0)
    .sort('MONTO_FRAUDE', descending=True)
    .head(20)
)

In [ ]:
# Scatter velocidad vs monto fraude
vel_pd = vel_cuenta.filter(pl.col('FRAUDES') > 0).to_pandas()

scatter_vel = (
    alt.Chart(vel_pd)
    .mark_circle(opacity=0.6)
    .encode(
        x     = alt.X('TRX_POR_DIA:Q', title='TRX promedio por día'),
        y     = alt.Y('MONTO_FRAUDE:Q', title='Monto Fraude (S/)',
                       axis=alt.Axis(format=',.0f')),
        size  = alt.Size('TOTAL_TRX:Q', title='Total TRX', legend=None),
        color = alt.Color('N_DESTINOS_DISTINTOS:Q',
                           scale=alt.Scale(scheme='reds'),
                           title='N° destinos distintos'),
        tooltip=['CUENTA:N',
                 alt.Tooltip('TRX_POR_DIA:Q', format='.2f'),
                 alt.Tooltip('MONTO_FRAUDE:Q', format=',.2f'),
                 alt.Tooltip('FRAUDES:Q'),
                 alt.Tooltip('N_DESTINOS_DISTINTOS:Q')]
    )
    .properties(width=550, height=300,
                title='Velocidad vs Monto Fraude por Cuenta — Hover para detalles')
)
scatter_vel

In [ ]:
# ── VERIFICACIÓN: Ver trx de cuentas con alta velocidad ───────────────────────
UMBRAL_TRX_DIA = 3   # ← cambia este umbral para explorar

cuentas_alta_vel = (
    vel_cuenta
    .filter((pl.col('TRX_POR_DIA') > UMBRAL_TRX_DIA) & (pl.col('FRAUDES') > 0))
    ['CUENTA'].to_list()
)
print(f'Cuentas con >{UMBRAL_TRX_DIA} trx/día Y con fraude: {len(cuentas_alta_vel)}')

trx_alta_vel = (
    df
    .filter(pl.col('CUENTA').is_in(cuentas_alta_vel))
    .select(['CUENTA','CUENTA_DESTINO','FECHA_TRX','HORA_TRX',
             'MONTO','SALDO_DISPONIBLE','INDICADOR','CANAL',
             'NOMBRE_BENEFICIARIO','FLAG_FRAUDE','FLAG_VACIADO_CUENTA'])
    .sort(['CUENTA','FECHA_TRX','HORA_TRX'])
)
print(f'Total transacciones de esas cuentas: {trx_alta_vel.shape[0]:,}')
trx_alta_vel.head(30)

## 7.1 Cuenta Destino — ¿A dónde van los fraudes?

In [ ]:
top_destino = (
    df
    .filter((pl.col('INDICADOR')=='F') & pl.col('CUENTA_DESTINO').is_not_null())
    .group_by('CUENTA_DESTINO')
    .agg([
        pl.len().alias('N_FRAUDES'),
        pl.col('CUENTA').n_unique().alias('N_CUENTAS_ORIGEN'),
        pl.col('MONTO').sum().round(2).alias('MONTO_FRAUDE'),
        pl.col('FECHA_TRX').min().alias('PRIMERA_TRX'),
        pl.col('FECHA_TRX').max().alias('ULTIMA_TRX'),
    ])
    .sort('MONTO_FRAUDE', descending=True)
)

print(f'Cuentas destino distintas en fraudes: {top_destino.shape[0]:,}')
print('Top 20 cuentas destino por monto recibido:')
print(top_destino.head(20))

---
# 8. Red de Fraude — Grafo de Beneficiarios

In [ ]:
edges_fraude = (
    df
    .filter(
        (pl.col('INDICADOR') == 'F') &
        pl.col('NOMBRE_BENEFICIARIO').is_not_null() &
        (pl.col('NOMBRE_BENEFICIARIO').str.len_chars() > 0)
    )
    .select([
        pl.col('CUENTA').alias('from'),
        pl.col('NOMBRE_BENEFICIARIO').alias('to'),
        pl.col('MONTO').alias('monto')
    ])
)

# Beneficiarios que reciben desde 2+ cuentas
bene_multi = (
    edges_fraude
    .group_by('to')
    .agg(pl.col('from').n_unique().alias('n'))
    .filter(pl.col('n') >= 2)
)
edges_red = edges_fraude.filter(pl.col('to').is_in(bene_multi['to']))

if edges_red.shape[0] > 0:
    G = nx.DiGraph()
    for row in edges_red.iter_rows(named=True):
        G.add_edge(row['from'], row['to'], weight=row['monto'])

    # Tipo de nodo
    nodos_origen = set(edges_red['from'].to_list())
    nodos_bene   = set(edges_red['to'].to_list())

    pos = nx.spring_layout(G, seed=42, k=2)

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_facecolor(C_BG)
    fig.patch.set_facecolor(C_BG)

    nx.draw_networkx_nodes(G, pos, nodelist=list(nodos_origen),
                           node_color=C_OK, node_size=200, alpha=0.8, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=list(nodos_bene),
                           node_color=C_FRAUD, node_size=500, alpha=0.9, ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color=C_ACCENT,
                           arrows=True, arrowsize=15,
                           width=1.5, alpha=0.6, ax=ax)
    nx.draw_networkx_labels(G, pos,
                            labels={n: n for n in nodos_bene},
                            font_size=7, font_color='#333', ax=ax)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color=C_OK,    label='Cuenta Origen'),
        Patch(color=C_FRAUD, label='Beneficiario (recibe fraude)'),
    ], loc='upper left')
    ax.set_title('Red de Fraude — Beneficiarios que reciben desde múltiples cuentas',
                 fontweight='bold', fontsize=13, color='#212529')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('ℹ️  No hay suficientes fraudes con beneficiario para el grafo.')

---
# 9. Propuesta de Reglas — Evaluación sobre volumen completo

In [ ]:
def evaluar_regla(nombre: str, mascara: pl.Expr) -> dict:
    sub       = df.filter(mascara)
    n_alertas = sub.shape[0]
    n_fraudes = sub.filter(pl.col('INDICADOR')=='F').shape[0]
    n_normales= sub.filter(pl.col('INDICADOR')=='N').shape[0]
    n_pend    = sub.filter(pl.col('INDICADOR')=='P').shape[0]
    monto_cap = sub.filter(pl.col('INDICADOR')=='F')['MONTO'].sum()
    return {
        'Regla'       : nombre,
        'Alertas'     : n_alertas,
        'Fraudes_F'   : n_fraudes,
        'Normales_N'  : n_normales,
        'Pendientes_P': n_pend,
        'Monto_Cap'   : round(monto_cap, 2) if monto_cap else 0,
        'Precision'   : round(n_fraudes/n_alertas*100, 2) if n_alertas>0 else 0,
        'Recall'      : round(n_fraudes/total_fraud*100, 2) if total_fraud>0 else 0,
        'Pct_Volumen' : round(n_alertas/df.shape[0]*100, 2),
    }

REGLAS = [
    ('R01: Monto Alto (≥5,000)',             pl.col('MONTO') >= 5000),
    ('R02: Monto Crítico (≥10,000)',         pl.col('MONTO') >= 10000),
    ('R03: Velocidad Alta (>3 trx/día)',     pl.col('FLAG_VEL_ALTA') == 1),
    ('R04: Anomalía Monto (|z|>2)',          pl.col('FLAG_ANOMALIA_MONTO') == 1),
    ('R05: Madrugada (0-5h)',                pl.col('FRANJA_HORARIA') == 'Madrugada'),
    ('R06: Vaciado cuenta (≥80%)',           pl.col('FLAG_VACIADO_CUENTA') == 1),
    ('R07: Cuenta nueva (≤3 trx previas)',   pl.col('FLAG_CUENTA_NUEVA') == 1),
    ('R08: Beneficiario nulo',               pl.col('FLAG_BENE_NULO') == 1),
    ('R09: Dispositivo multiusuario (ATO)',  pl.col('FLAG_DISPOSITIVO_MULTIUSUARIO') == 1),
    ('R10: Vaciado + Madrugada',             (pl.col('FLAG_VACIADO_CUENTA')==1) & (pl.col('FRANJA_HORARIA')=='Madrugada')),
    ('R11: MOT + Madrugada + Vaciado',       (pl.col('CANAL')=='MOT') & (pl.col('FRANJA_HORARIA')=='Madrugada') & (pl.col('FLAG_VACIADO_CUENTA')==1)),
    ('R12: Cuenta nueva + Monto Alto',       (pl.col('FLAG_CUENTA_NUEVA')==1) & (pl.col('MONTO')>=5000)),
    ('R13: ATO + Monto Alto',                (pl.col('FLAG_DISPOSITIVO_MULTIUSUARIO')==1) & (pl.col('MONTO')>=5000)),
]

df_reglas = pl.DataFrame([evaluar_regla(n, m) for n, m in REGLAS]).sort('Precision', descending=True)
print('Evaluación de Reglas sobre TODO el volumen (incluyendo N):')
print(df_reglas)

In [ ]:
# Gráfico Precisión vs Recall
df_reg_pd = df_reglas.to_pandas()

prec_bars = (
    alt.Chart(df_reg_pd)
    .mark_bar(color=C_FRAUD, opacity=0.85)
    .encode(
        y       = alt.Y('Regla:N', sort='-x', title=None),
        x       = alt.X('Precision:Q', title='Precisión (%)'),
        tooltip = ['Regla:N',
                   alt.Tooltip('Precision:Q', format='.2f'),
                   alt.Tooltip('Recall:Q', format='.2f'),
                   alt.Tooltip('Alertas:Q', format=','),
                   alt.Tooltip('Fraudes_F:Q'),
                   alt.Tooltip('Normales_N:Q', format=','),
                   alt.Tooltip('Monto_Cap:Q', format=',.2f')]
    )
    .properties(width=350, height=320, title='Precisión por Regla')
)

recall_bars = (
    alt.Chart(df_reg_pd)
    .mark_bar(color=C_ACCENT, opacity=0.85)
    .encode(
        y       = alt.Y('Regla:N', sort=alt.EncodingSortField('Precision', order='descending'), title=None),
        x       = alt.X('Recall:Q', title='Recall (%)'),
        tooltip = ['Regla:N',
                   alt.Tooltip('Precision:Q', format='.2f'),
                   alt.Tooltip('Recall:Q', format='.2f')]
    )
    .properties(width=350, height=320, title='Recall por Regla')
)

(prec_bars | recall_bars)

---
# 10. Export — Parquet para Power BI

In [ ]:
OUTPUT_DIR = BASE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset completo con todos los flags → Power BI ───────────────────────────
df.write_parquet(OUTPUT_DIR / 'transferencias_terceros_FLAGS.parquet')

# ── Solo fraudes confirmados → revisión manual ────────────────────────────────
(
    df.filter(pl.col('INDICADOR') == 'F')
      .sort('MONTO', descending=True)
      .to_pandas()
      .to_excel(OUTPUT_DIR / 'fraudes_transferencias_terceros.xlsx', index=False)
)

# ── Tabla de reglas → para presentar al especialista ─────────────────────────
df_reglas.to_pandas().to_excel(OUTPUT_DIR / 'evaluacion_reglas.xlsx', index=False)

# ── Top cuentas destino ───────────────────────────────────────────────────────
top_destino.to_pandas().to_excel(OUTPUT_DIR / 'cuentas_destino_fraude.xlsx', index=False)

# ── Resumen de velocidad por cuenta ──────────────────────────────────────────
vel_cuenta.to_pandas().to_excel(OUTPUT_DIR / 'velocidad_por_cuenta.xlsx', index=False)

print('✅ Archivos exportados:')
print(f'   → transferencias_terceros_FLAGS.parquet  ← Power BI')
print(f'   → fraudes_transferencias_terceros.xlsx')
print(f'   → evaluacion_reglas.xlsx')
print(f'   → cuentas_destino_fraude.xlsx')
print(f'   → velocidad_por_cuenta.xlsx')
print(f'\n📂 Ruta: {OUTPUT_DIR}')

# Verificar columnas que llegan a Power BI
flags_pbi = [c for c in df.columns if c.startswith('FLAG_')]
print(f'\n📊 Flags disponibles en Power BI ({len(flags_pbi)}):')
for f in flags_pbi:
    print(f'   · {f}')

---
## 📝 Conclusiones

| Dimensión | Hallazgo clave | Sección |
|---|---|---|
| **Temporalidad** | Madrugada concentra mayor Fraud Rate + LIFT | 5.3 |
| **Canal** | MOT en madrugada = combinación más riesgosa | 5.2 |
| **Monto** | Árbol CART define cortes óptimos estadísticos | 4.3 |
| **Saldo** | Ratio ≥80% discrimina vaciado de cuenta | 4.4 |
| **Historia** | CC_K05 ≤3 → posibles cuentas mula | 2.5 |
| **Dispositivo** | Hash multiusuario → ATO confirmado | 2.8 |
| **Red** | Beneficiarios concentrados → fraude organizado | 8 |
| **Reglas** | Evaluar siempre sobre volumen completo incluyendo N | 9 |

> **Próximo paso:** Conectar `transferencias_terceros_FLAGS.parquet` a Power BI  
> y construir el dashboard con los flags como segmentadores.

---
*Generado automáticamente · Prevención de Fraude · Scotiabank Perú*